<span style="color: #6a737d; font-family: monospace;">
Created on Mon Jan 27 2025 16:38:44<br>
Author: Mukai (Tom Notch) Yu<br>
Email: mukaiy@andrew.cmu.edu<br>
Affiliation: Carnegie Mellon University, Robotics Institute<br>
<br>
Copyright Ⓒ 2025 Mukai (Tom Notch) Yu<br>
</span>

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# %cd $USF_PROJECT_DIRECTORY doesn't work here because it's set by os.environ, not before the notebook starts
%cd ../..
%load_ext autoreload
%autoreload 2

import math
import os.path as osp

import cv2
import torch
from matplotlib import pyplot as plt

from usf.network.layer.spherical.circle_pool import CirclePool
from usf.network.layer.spherical.generic_spherical_cnn import GenericSphericalConv
from usf.utils.files import read_file
from usf.utils.spherical_image import BatchSphericalImage, SphericalImage
from usf.visualization.spherical_layer import visualize_spherical_channels
from usf.visualization.spherical_projection import visualize_spherical_image

In [ ]:
dtype = torch.float32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
factory_kwargs = {"device": device, "dtype": dtype}

In [ ]:
CURRENT_DIR = osp.dirname(osp.realpath("__file__"))
CAMERA_CFG_PATH = osp.join(CURRENT_DIR, "config/wildfire/subcanopy/subcanopy.yaml")
INPUT_IMAGE_PATH = osp.join(
    CURRENT_DIR, "config/wildfire/subcanopy/sensors/camera_0_sample.png"
)

In [ ]:
# Load the camera configuration
camera_config = read_file(CAMERA_CFG_PATH)["sensor_configs"]["rgb_0"]

# lens normal map
lens_normal_map = camera_config["lens_normal_map"]

# mask
mask = camera_config["mask"]

In [ ]:
# Load example image
input_image = read_file(INPUT_IMAGE_PATH)
input_image = cv2.resize(
    input_image, (camera_config["image_width"], camera_config["image_height"])
)

# Display one of the images
plt.axis("off")
plt.imshow(
    input_image,
)
plt.show()

In [ ]:
mask_gray = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
_, binary_mask = cv2.threshold(mask_gray, 128, 255, cv2.THRESH_BINARY)
binary_mask = binary_mask.astype(bool)

image_color = input_image[binary_mask].reshape(-1, 3)
image_vector = lens_normal_map[binary_mask].reshape(-1, 3)
spherical_image = SphericalImage(value=image_color, vector=image_vector)

In [ ]:
visualize_spherical_image(spherical_image, point_size=5)

## Create a BatchSphericalImage

Stack 3 copies of spherical_image to mimic batched input

In [ ]:
batch_spherical_image = BatchSphericalImage([spherical_image] * 3).to(**factory_kwargs)
str(batch_spherical_image)

# Spherical CNN with Generic Weighting Function

In [ ]:
spherical_conv = GenericSphericalConv(
    backend="spherical",
    in_channels=3,
    out_channels=8,
    radius=torch.pi / 128,
    weighting_function_config={
        "distance": {
            "function": "continuous",
            "hidden_dims": [8, 8],
            "activation": "ReLU",
            "embedding": {"type": "cosine", "L": 6},
            # "function": "discrete",
            # "num_slices": 3,
        },
        "direction": {
            "function": "continuous",
            "hidden_dims": [8, 8],
            "activation": "ReLU",
            "embedding": {"type": "fourier", "L": 6, "include_x": False},
            # "function": "discrete",
            # "num_slices": 6,
        },
    },
    resolution_factor=0.1,
    location_sampler="healpix",
).to(
    **factory_kwargs
)  # pyright: ignore[reportAbstractUsage]

In [ ]:
spherical_conv_batch_spherical_image = spherical_conv(batch_spherical_image)

In [ ]:
first_value = spherical_conv_batch_spherical_image[0].value

all(
    torch.allclose(first_value, spherical_image.value, atol=1e-5)
    for spherical_image in spherical_conv_batch_spherical_image
)

In [ ]:
spherical_conv.report_metrics()

In [ ]:
visualize_spherical_image(
    visualize_spherical_channels(spherical_conv_batch_spherical_image)[0],
    point_size=5,
    fps=1,
)

## Visualize the Kernel

In [ ]:
spherical_conv_spherical_image = spherical_conv.visualize_kernel(num_display_kernel=4)

In [ ]:
visualize_spherical_image(spherical_conv_spherical_image, point_size=5)

In [ ]:
import math

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import torch

radius = spherical_conv.radius
r = radius
k0 = spherical_conv.weighting_function.branches["distance"].embedder.basis
L = spherical_conv.weighting_function.branches["distance"].embedder.L
max_freq = L * k0

N = max(100000, int(max_freq * 4 * 10))
x = torch.linspace(-torch.pi, 3 * torch.pi, N).unsqueeze(-1)  # (N, 1)

with torch.no_grad():
    distance_fn = spherical_conv.weighting_function.branches["distance"]
    direction_fn = spherical_conv.weighting_function.branches["direction"]
    _device = next(distance_fn.parameters()).device
    y_dist = distance_fn(x.to(_device))  # (N, F, 1)
    y_dir = direction_fn(x.to(_device))  # (N, F, 1)

x_np = x.squeeze(-1).numpy()
y_dist_np = y_dist.squeeze(-1).cpu().numpy()  # (N, F)
y_dir_np = y_dir.squeeze(-1).cpu().numpy()  # (N, F)

num_functions = y_dist_np.shape[1]
num_cols = 4
num_rows = math.ceil(num_functions / num_cols)


def pi_formatter(val, _pos):
    if val == 0:
        return "0"
    coeff = val / math.pi
    if abs(coeff - round(coeff)) < 1e-3:
        c = int(round(coeff))
        if c == 1:
            return "π"
        if c == -1:
            return "−π"
        return f"{c}π"
    return f"{coeff:.1f}π"


# --- Distance function ---
fig, axes = plt.subplots(
    num_rows, num_cols, figsize=(4 * num_cols, 3 * num_rows), squeeze=False
)
fig.suptitle(
    f"Distance weighting functions (radius={radius:.4f}, r={r:.4f}, k₀={k0:.0f}, L={L})"
)
for idx in range(num_functions):
    ax = axes[idx // num_cols, idx % num_cols]
    ax.plot(x_np, y_dist_np[:, idx], linewidth=0.5)
    ax.axvline(r, color="red", linestyle="--", linewidth=0.8, label=f"r={r:.4f}")
    ax.axvline(-r, color="red", linestyle="--", linewidth=0.8)
    for k in range(-1, 4):
        ax.axvline(
            k * math.pi,
            color="green",
            linestyle=":",
            linewidth=0.7,
            label="kπ" if k == 0 else None,
        )
    ax.set_title(f"f_{idx}", fontsize=9)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(base=math.pi))
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(pi_formatter))
    ax.tick_params(labelsize=7)
for idx in range(num_functions, num_rows * num_cols):
    axes[idx // num_cols, idx % num_cols].set_visible(False)
axes[0, 0].legend(fontsize=7, loc="upper right")
fig.tight_layout()
plt.show()

# --- Distance function (zoomed into kernel support) ---
half_period = math.pi / k0  # symmetric axes of cos(k0 * d) at n * π/k0
zoom_margin = half_period * 0.5
zoom_lo, zoom_hi = -zoom_margin, half_period * 3
zoom_mask = (x_np >= zoom_lo) & (x_np <= zoom_hi)
x_zoom = x_np[zoom_mask]
y_dist_zoom = y_dist_np[zoom_mask]

fig, axes = plt.subplots(
    num_rows, num_cols, figsize=(4 * num_cols, 3 * num_rows), squeeze=False
)
fig.suptitle(
    f"Distance weighting functions — zoomed (r={r:.4f}, π/k₀={half_period:.4f})"
)
for idx in range(num_functions):
    ax = axes[idx // num_cols, idx % num_cols]
    ax.plot(x_zoom, y_dist_zoom[:, idx], linewidth=1.0)
    ax.axvline(0, color="blue", linestyle="-", linewidth=0.8, label="0")
    ax.axvline(r, color="red", linestyle="--", linewidth=0.8, label=f"r={r:.4f}")
    n = 1
    while n * half_period <= zoom_hi:
        ax.axvline(
            n * half_period,
            color="green",
            linestyle=":",
            linewidth=0.8,
            label=f"nπ/k₀ (n={n})" if idx == 0 else None,
        )
        n += 1
    ax.set_title(f"f_{idx}", fontsize=9)
    ax.tick_params(labelsize=7)
for idx in range(num_functions, num_rows * num_cols):
    axes[idx // num_cols, idx % num_cols].set_visible(False)
axes[0, 0].legend(fontsize=7, loc="upper right")
fig.tight_layout()
plt.show()

# --- Direction function ---
fig, axes = plt.subplots(
    num_rows, num_cols, figsize=(4 * num_cols, 3 * num_rows), squeeze=False
)
fig.suptitle("Direction weighting functions")
for idx in range(num_functions):
    ax = axes[idx // num_cols, idx % num_cols]
    ax.plot(x_np, y_dir_np[:, idx], linewidth=0.5)
    for k in range(-1, 4):
        ax.axvline(
            k * math.pi,
            color="green",
            linestyle=":",
            linewidth=0.7,
            label="kπ" if k == 0 else None,
        )
    ax.set_title(f"f_{idx}", fontsize=9)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(base=math.pi))
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(pi_formatter))
    ax.tick_params(labelsize=7)
for idx in range(num_functions, num_rows * num_cols):
    axes[idx // num_cols, idx % num_cols].set_visible(False)
axes[0, 0].legend(fontsize=7, loc="upper right")
fig.tight_layout()
plt.show()

## Visualize Computation Graph

In [ ]:
from torchviz import make_dot

make_dot(
    spherical_conv_batch_spherical_image.batch_value,
    params=dict(spherical_conv.named_parameters()),
).render("spherical_conv_computation_graph", format="pdf")

# Circle CNN

### Instantiate a layer with weight = 1 and bias = 0

Effectively an average operator

In [ ]:
circle_conv = GenericSphericalConv(
    backend="circle",
    in_channels=3,
    out_channels=8,
    radius=torch.pi / 128,
    weighting_function_config={
        "distance": {
            "function": "discrete",
            "num_slices": 5,
        },
    },
    resolution_factor=0.1,
    location_sampler="healpix",
).to(**factory_kwargs)

In [ ]:
# experiment with weight = 1 and bias = 0, effectively an average operator
with torch.no_grad():
    circle_conv.weight.copy_(torch.ones_like(circle_conv.weight))
    circle_conv.bias.copy_(torch.zeros_like(circle_conv.bias))

## Forward

Will be slow for new vector since caching takes time

In [ ]:
circle_batch_spherical_image = circle_conv(batch_spherical_image)

Check that the output of identical input spherical images are identical

In [ ]:
first_value = circle_batch_spherical_image[0].value

all(
    torch.allclose(first_value, spherical_image.value)
    for spherical_image in circle_batch_spherical_image
)

In [ ]:
circle_conv.report_metrics()

## Visualize Output Image

In [ ]:
visualize_spherical_image(
    visualize_spherical_channels(circle_batch_spherical_image)[0], point_size=5, fps=1
)

## Visualize the Kernel

In [ ]:
circle_conv_spherical_image = circle_conv.visualize_kernel(num_display_kernel=4)

In [ ]:
visualize_spherical_image(circle_conv_spherical_image, point_size=5)

## Visualize Computation Graph

In [ ]:
from torchviz import make_dot

make_dot(
    circle_batch_spherical_image.batch_value,
    params=dict(wave_conv.named_parameters()),
).render("circle_conv_computation_graph", format="pdf")

# Circle Pool

In [ ]:
circle_pool = CirclePool(
    pool_type="max",
    radius=torch.pi / 100,
    resolution_factor=0.05,
    location_sampler="healpix",
).to(**factory_kwargs)

## Forward

In [ ]:
pooled_batch_spherical_image = circle_pool(batch_spherical_image)

Check that the output of identical input spherical images are identical

In [ ]:
first_value = pooled_batch_spherical_image[0].value

all(
    torch.allclose(first_value, spherical_image.value)
    for spherical_image in pooled_batch_spherical_image
)

In [ ]:
circle_pool.report_metrics()

In [ ]:
visualize_spherical_image(pooled_batch_spherical_image, point_size=5)

## Visualize the Kernel

In [ ]:
circle_pool_spherical_image = circle_pool.visualize_kernel(num_display_kernel=2)

In [ ]:
visualize_spherical_image(circle_pool_spherical_image, point_size=5)

## Visualize Computation Graph

In [ ]:
from torchviz import make_dot

make_dot(
    pooled_batch_spherical_image.batch_value,
    params=dict(circle_pool.named_parameters()),
).render("circle_pool_computation_graph", format="pdf")